# VLM Disaster Analyzer — Video Dataset Manager

**Workflow**
1. Mount Google Drive
2. Create / verify the `DisasterVideoDataset/` folder structure
3. Load video paths from Drive folders
4. Extract per-video metadata (frame count, duration, resolution)
5. Build a pandas DataFrame and export to `dataset_manifest.xlsx`

> **No datasets are downloaded automatically.**  
> Manually upload videos to the class folders in Google Drive, then run this notebook.

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted at /content/drive')

## 2 · Configuration

In [ ]:
import os

# ── Paths ──────────────────────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive'
DATASET_ROOT = os.path.join(DRIVE_ROOT, 'DisasterVideoDataset')
MANIFEST_OUT = os.path.join(DATASET_ROOT, 'dataset_manifest.xlsx')

# ── Disaster classes (match VLM Disaster Analyzer label set) ────────────────
CLASSES = [
    'Earthquake',
    'Flood',
    'Wildfire',
    'UrbanFire',
    'Landslide',
    'Cyclone',
    'Drought',
]

# ── Accepted video extensions ───────────────────────────────────────────────
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.m4v', '.flv'}

# ── Frame sampling ─────────────────────────────────────────────────────────
DEFAULT_SAMPLE_N = 8   # frames to sample per video for downstream VLM inference

print(f'Dataset root : {DATASET_ROOT}')
print(f'Manifest out : {MANIFEST_OUT}')
print(f'Classes      : {CLASSES}')

## 3 · Create folder structure (safe — no-op if already exists)

In [ ]:
def create_dataset_folders(root: str, classes: list[str]) -> None:
    """Create root + one sub-folder per class. Safe to re-run."""
    os.makedirs(root, exist_ok=True)
    for cls in classes:
        path = os.path.join(root, cls)
        os.makedirs(path, exist_ok=True)
        status = 'exists' if os.path.isdir(path) else 'created'
        print(f'  [{status:^7}]  {path}')

print('Creating / verifying folder structure...')
create_dataset_folders(DATASET_ROOT, CLASSES)
print('Done.')

## 4 · Verify structure and count videos per class

In [ ]:
def count_videos(folder: str, extensions: set[str]) -> int:
    """Count video files directly inside a folder (non-recursive)."""
    if not os.path.isdir(folder):
        return 0
    return sum(
        1 for f in os.listdir(folder)
        if os.path.splitext(f)[1].lower() in extensions
    )

print(f'\n{"Class":<20} {"Folder exists":<16} {"Videos"}')
print('-' * 48)

total = 0
for cls in CLASSES:
    folder  = os.path.join(DATASET_ROOT, cls)
    exists  = os.path.isdir(folder)
    n       = count_videos(folder, VIDEO_EXTENSIONS) if exists else 0
    total  += n
    marker  = '✓' if exists else '✗'
    print(f'{cls:<20} {marker:<16} {n}')

print('-' * 48)
print(f'{"TOTAL":<20} {"":<16} {total}')

if total == 0:
    print('\n⚠  No videos found yet.')
    print('   Upload videos to the class folders in Google Drive,')
    print('   then re-run cells 4 onwards.')
else:
    print(f'\n✓  {total} video(s) ready for processing.')

## 5 · Install / import dependencies

In [ ]:
# OpenCV is pre-installed on Colab; openpyxl is needed for .xlsx export
!pip install -q openpyxl

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional
from tqdm.notebook import tqdm

print('Dependencies ready.')

## 6 · Helper functions

In [ ]:
# ---------------------------------------------------------------------------
# load_video_paths
# ---------------------------------------------------------------------------

def load_video_paths(
    root: str,
    classes: list[str],
    extensions: set[str] = VIDEO_EXTENSIONS,
) -> list[dict]:
    """
    Walk dataset root and collect every video file with its class label.

    Returns a list of dicts:
        [{'video_path': str, 'class_label': str}, ...]
    """
    records = []
    for cls in classes:
        folder = os.path.join(root, cls)
        if not os.path.isdir(folder):
            continue
        for fname in sorted(os.listdir(folder)):
            if os.path.splitext(fname)[1].lower() in extensions:
                records.append({
                    'video_path':  os.path.join(folder, fname),
                    'class_label': cls,
                })
    return records


# ---------------------------------------------------------------------------
# extract_frames
# ---------------------------------------------------------------------------

def extract_frames(
    video_path: str,
    max_frames: Optional[int] = None,
) -> list[np.ndarray]:
    """
    Read every frame (or up to max_frames) from a video file.

    Returns a list of BGR numpy arrays (H, W, 3).
    Returns [] if the file cannot be opened.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'  [WARN] Cannot open: {video_path}')
        return []

    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(frame)
        if max_frames and len(frames) >= max_frames:
            break

    cap.release()
    return frames


# ---------------------------------------------------------------------------
# sample_n_frames
# ---------------------------------------------------------------------------

def sample_n_frames(
    video_path: str,
    n: int = DEFAULT_SAMPLE_N,
    strategy: str = 'uniform',
) -> list[np.ndarray]:
    """
    Sample exactly n frames from a video.

    strategy:
      'uniform'  — evenly spaced across the full duration (default)
      'first'    — first n frames
      'middle'   — n frames centred on the middle of the video

    Returns a list of BGR numpy arrays. May return fewer than n
    if the video has fewer frames.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'  [WARN] Cannot open: {video_path}')
        return []

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []

    n = min(n, total)

    if strategy == 'first':
        indices = list(range(n))
    elif strategy == 'middle':
        mid   = total // 2
        start = max(0, mid - n // 2)
        indices = list(range(start, min(start + n, total)))
    else:  # uniform
        indices = [int(i * (total - 1) / (n - 1)) for i in range(n)] if n > 1 else [total // 2]

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if ok:
            frames.append(frame)

    cap.release()
    return frames


# ---------------------------------------------------------------------------
# _video_metadata  (internal)
# ---------------------------------------------------------------------------

def _video_metadata(video_path: str) -> dict:
    """
    Extract metadata from a single video file using OpenCV.

    Returns a dict with:
        frame_count, fps, duration_seconds, width, height, file_size_mb
    Returns None values if the file cannot be opened.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return {
            'frame_count': None, 'fps': None,
            'duration_seconds': None, 'width': None,
            'height': None, 'file_size_mb': None,
        }

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps         = cap.get(cv2.CAP_PROP_FPS)
    width       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    duration = round(frame_count / fps, 2) if fps and fps > 0 else None
    size_mb  = round(os.path.getsize(video_path) / (1024 ** 2), 2)

    return {
        'frame_count':      frame_count,
        'fps':              round(fps, 2),
        'duration_seconds': duration,
        'width':            width,
        'height':           height,
        'file_size_mb':     size_mb,
    }


# ---------------------------------------------------------------------------
# build_video_manifest
# ---------------------------------------------------------------------------

def build_video_manifest(
    root: str,
    classes: list[str],
    extensions: set[str] = VIDEO_EXTENSIONS,
) -> pd.DataFrame:
    """
    Walk all class folders and build a pandas DataFrame describing every video.

    Columns:
        video_path, class_label, frame_count, fps,
        duration_seconds, width, height, file_size_mb
    """
    records = load_video_paths(root, classes, extensions)

    if not records:
        print('No videos found — returning empty DataFrame.')
        return pd.DataFrame(columns=[
            'video_path', 'class_label', 'frame_count', 'fps',
            'duration_seconds', 'width', 'height', 'file_size_mb',
        ])

    rows = []
    for rec in tqdm(records, desc='Reading video metadata'):
        meta = _video_metadata(rec['video_path'])
        rows.append({**rec, **meta})

    df = pd.DataFrame(rows)[[
        'video_path', 'class_label', 'frame_count', 'fps',
        'duration_seconds', 'width', 'height', 'file_size_mb',
    ]]
    return df


print('Helper functions defined.')

## 7 · Build the manifest DataFrame

In [ ]:
df = build_video_manifest(DATASET_ROOT, CLASSES)

print(f'\nManifest shape: {df.shape[0]} videos × {df.shape[1]} columns')
df.head(10)

## 8 · Dataset statistics

In [ ]:
if df.empty:
    print('No videos in the dataset yet. Upload videos to Drive and re-run.')
else:
    print('=' * 56)
    print('  VLM DISASTER ANALYZER — VIDEO DATASET STATISTICS')
    print('=' * 56)

    # ── Per-class summary ─────────────────────────────────────────────────
    summary = (
        df.groupby('class_label')
        .agg(
            videos        = ('video_path',      'count'),
            total_frames  = ('frame_count',     'sum'),
            total_sec     = ('duration_seconds','sum'),
            avg_duration  = ('duration_seconds','mean'),
            total_mb      = ('file_size_mb',    'sum'),
        )
        .reindex(CLASSES)   # preserve class order
        .fillna(0)
        .astype({'videos': int, 'total_frames': int})
    )
    summary['avg_duration'] = summary['avg_duration'].round(1)
    summary['total_mb']     = summary['total_mb'].round(1)
    summary['total_sec']    = summary['total_sec'].round(1)

    print('\nPer-class breakdown:')
    print(summary.to_string())

    # ── Totals ────────────────────────────────────────────────────────────
    print('\n' + '-' * 56)
    print(f"  Total videos      : {len(df)}")
    print(f"  Total frames      : {int(df['frame_count'].sum()):,}")
    print(f"  Total duration    : {df['duration_seconds'].sum() / 60:.1f} minutes")
    print(f"  Total size        : {df['file_size_mb'].sum():.1f} MB")
    print(f"  Classes covered   : {df['class_label'].nunique()} / {len(CLASSES)}")

    # ── Class balance ─────────────────────────────────────────────────────
    missing = [c for c in CLASSES if c not in df['class_label'].values]
    if missing:
        print(f"\n  ⚠  Empty classes : {', '.join(missing)}")
    else:
        print('\n  ✓  All classes have at least one video.')

    print('=' * 56)

## 9 · Export manifest to Excel

In [ ]:
if df.empty:
    print('Manifest is empty — nothing to export.')
else:
    with pd.ExcelWriter(MANIFEST_OUT, engine='openpyxl') as writer:

        # Sheet 1: full manifest
        df.to_excel(writer, sheet_name='Manifest', index=False)

        # Sheet 2: per-class summary
        summary_export = (
            df.groupby('class_label')
            .agg(
                videos            = ('video_path',      'count'),
                total_frames      = ('frame_count',     'sum'),
                total_duration_s  = ('duration_seconds','sum'),
                avg_duration_s    = ('duration_seconds','mean'),
                total_size_mb     = ('file_size_mb',    'sum'),
            )
            .reindex(CLASSES)
            .fillna(0)
            .reset_index()
        )
        summary_export.to_excel(writer, sheet_name='Class Summary', index=False)

    print(f'✓  Manifest exported to:')
    print(f'   {MANIFEST_OUT}')
    print(f'   Sheets: "Manifest" ({len(df)} rows)  |  "Class Summary" ({len(summary_export)} rows)')

## 10 · Frame sampling demo

Demonstrates `sample_n_frames()` on the first video found in the dataset.  
This is the function your VLM inference pipeline calls before sending frames to CLIP or Qwen.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

if df.empty:
    print('No videos available for frame demo.')
else:
    sample_row  = df.iloc[0]
    sample_path = sample_row['video_path']
    sample_cls  = sample_row['class_label']

    print(f'Sampling from: {os.path.basename(sample_path)}  [{sample_cls}]')

    frames = sample_n_frames(sample_path, n=DEFAULT_SAMPLE_N, strategy='uniform')
    print(f'Sampled {len(frames)} frames')

    if frames:
        cols = min(4, len(frames))
        rows = (len(frames) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 2.8))
        axes = np.array(axes).flatten()

        for i, frame in enumerate(frames):
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            axes[i].imshow(rgb)
            axes[i].set_title(f'Frame {i+1}', fontsize=9)
            axes[i].axis('off')

        for j in range(len(frames), len(axes)):
            axes[j].set_visible(False)

        fig.suptitle(
            f'{sample_cls} — {os.path.basename(sample_path)}\n'
            f'{len(frames)} uniformly sampled frames  |  '
            f'total: {int(sample_row["frame_count"])} frames  |  '
            f'{sample_row["duration_seconds"]}s',
            fontsize=10, y=1.02,
        )
        plt.tight_layout()
        plt.show()
    else:
        print('  Could not read frames from this file.')

## 11 · VLM inference integration stub

Shows how to wire the dataset helpers into the existing VLM Disaster Analyzer backend.
Replace the `TODO` blocks with your actual CLIP / Qwen calls once the backend is accessible from Colab.

In [ ]:
import tempfile
import io
from PIL import Image

# ── VLM backend URL ────────────────────────────────────────────────────────
# If the backend is exposed via ngrok, replace the URL below:
VLM_BACKEND_URL = 'http://localhost:8000'   # or 'https://<your-ngrok-id>.ngrok.io'


def frame_to_jpeg_bytes(frame: np.ndarray, quality: int = 90) -> bytes:
    """Convert a BGR numpy frame to JPEG bytes suitable for multipart upload."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=quality)
    return buf.getvalue()


def predict_clip_from_frame(frame: np.ndarray, backend_url: str = VLM_BACKEND_URL) -> dict:
    """
    Send a single frame to the VLM Disaster Analyzer CLIP endpoint.

    Returns the JSON response dict from POST /predict/clip.
    """
    import requests
    jpeg = frame_to_jpeg_bytes(frame)
    response = requests.post(
        f'{backend_url}/predict/clip',
        files={'file': ('frame.jpg', jpeg, 'image/jpeg')},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def analyze_video_with_clip(
    video_path: str,
    n_frames: int = DEFAULT_SAMPLE_N,
    backend_url: str = VLM_BACKEND_URL,
) -> pd.DataFrame:
    """
    Sample n_frames from a video, run each through CLIP, return a per-frame
    results DataFrame.

    Columns: frame_idx, disaster_type, confidence_score, confidence_level
    """
    frames  = sample_n_frames(video_path, n=n_frames, strategy='uniform')
    results = []

    for i, frame in enumerate(tqdm(frames, desc=os.path.basename(video_path), leave=False)):
        try:
            resp    = predict_clip_from_frame(frame, backend_url)
            metrics = resp.get('metrics', {})
            results.append({
                'frame_idx':       i,
                'disaster_type':   metrics.get('disaster_type',   'Unknown'),
                'confidence_score':metrics.get('confidence_score', 0.0),
                'confidence_level':metrics.get('confidence_level', ''),
            })
        except Exception as exc:
            results.append({
                'frame_idx':        i,
                'disaster_type':    'ERROR',
                'confidence_score': 0.0,
                'confidence_level': str(exc),
            })

    return pd.DataFrame(results)


print('VLM integration stubs defined.')
print(f'Backend URL : {VLM_BACKEND_URL}')
print('\nTo use:')
print('  results = analyze_video_with_clip(df.iloc[0]["video_path"])')
print('  print(results)')

## 12 · Quick reference

| Function | Purpose |
|---|---|
| `load_video_paths(root, classes)` | Returns `[{video_path, class_label}]` for all videos |
| `extract_frames(path, max_frames)` | Reads all (or up to N) frames as BGR arrays |
| `sample_n_frames(path, n, strategy)` | Samples exactly N frames — uniform / first / middle |
| `build_video_manifest(root, classes)` | Returns full metadata DataFrame |
| `analyze_video_with_clip(path, n)` | Runs CLIP on N sampled frames via backend API |

**Drive folder structure expected:**
```
MyDrive/
└── DisasterVideoDataset/
    ├── Earthquake/     ← drop .mp4/.avi/.mov files here
    ├── Flood/
    ├── Wildfire/
    ├── UrbanFire/
    ├── Landslide/
    ├── Cyclone/
    ├── Drought/
    └── dataset_manifest.xlsx   ← auto-generated by cell 9
```

**Accepted video formats:** `.mp4 .avi .mov .mkv .webm .m4v .flv`